# Recover `backbone.pt` from a `last_ckpt.pt`

Standalone converter for an interrupted SimCLR run that trained to the last epoch but died before writing `backbone.pt`. It rebuilds `backbone.pt` **from the checkpoint weights alone — no retraining, no dataset, no GPU, only `torch`**.

**How it works:** the checkpoint's `model` state_dict has keys prefixed `backbone.` and `projector.`. This splits them and repackages into the `backbone.pt` format the eval code expects (`kind`, `seed`, `backbone`, `projector`, `config.projector`, `final_loss`).

**Steps:** upload your `last_ckpt.pt` (and optionally `train_log.jsonl` to recover the final loss), set the four fields in the next cell, then run every cell top to bottom.

In [ ]:
# --- set these ---------------------------------------------------------
SEED      = 5                              # the seed this checkpoint belongs to
OUT_PATH  = "/kaggle/working/backbone.pt"  # writable output (/kaggle/input is read-only)

# Projector geometry from configs/base.yaml — same for every condition (nothing overrides it).
PROJECTOR = {"hidden_dim": 512, "out_dim": 128}

### Locating the checkpoint

Your checkpoint is attached as a **Kaggle dataset**, mounted read-only under `/kaggle/input/`. The next cell finds `last_ckpt.pt` there automatically (and `train_log.jsonl` if it's in the same folder). If more than one `last_ckpt.pt` is attached, it prints all of them so you can hardcode `CKPT_PATH`.

In [ ]:
# Auto-locate the checkpoint in the attached Kaggle dataset (read-only /kaggle/input).
import glob, os

hits = sorted(glob.glob("/kaggle/input/**/last_ckpt.pt", recursive=True))
assert hits, "no last_ckpt.pt under /kaggle/input — is the dataset attached? (Add Data on the right)"
if len(hits) > 1:
    print("multiple checkpoints found — set CKPT_PATH to the one you want:")
    for h in hits:
        print("   ", h)
CKPT_PATH = hits[0]

# train_log.jsonl in the same folder -> recover the real final_loss (else NaN).
_log = os.path.join(os.path.dirname(CKPT_PATH), "train_log.jsonl")
LOG_PATH = _log if os.path.exists(_log) else None

print("checkpoint:", CKPT_PATH)
print("log:       ", LOG_PATH)

In [ ]:
import json
import math
import torch

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if "backbone" in ckpt and "projector" in ckpt:
    raise SystemExit(f"{CKPT_PATH} already looks like a backbone.pt — nothing to convert.")
sd = ckpt["model"] if "model" in ckpt else ckpt

def _strip(key):
    return key[len("_orig_mod."):] if key.startswith("_orig_mod.") else key

backbone_sd, projector_sd = {}, {}
for k, v in sd.items():
    k = _strip(k)
    if k.startswith("backbone."):
        backbone_sd[k[len("backbone."):]] = v
    elif k.startswith("projector."):
        projector_sd[k[len("projector."):]] = v
assert backbone_sd and projector_sd, (
    "unexpected checkpoint layout — no backbone./projector. keys found in the model state_dict"
)

# Recover final_loss from the last train_log line, if a log was provided.
final_loss = float("nan")
if LOG_PATH:
    last = None
    with open(LOG_PATH) as fh:
        for line in fh:
            if line.strip():
                last = json.loads(line)
    if last and last.get("event") != "nan_abort":
        final_loss = float(last.get("loss", float("nan")))

out = {
    "kind": "simclr",
    "seed": SEED,
    "backbone": backbone_sd,
    "projector": projector_sd,
    "config": {"projector": PROJECTOR},
    "final_loss": final_loss,
}
torch.save(out, OUT_PATH)
print(
    f"wrote {OUT_PATH}: backbone={len(backbone_sd)} tensors, "
    f"projector={len(projector_sd)} tensors, epoch={ckpt.get('epoch', '?')}, "
    f"seed={SEED}, final_loss={final_loss}"
)

In [ ]:
# --- validate the file the eval code will actually read --------------------
chk = torch.load(OUT_PATH, map_location="cpu", weights_only=False)
assert chk["kind"] == "simclr"
assert chk["backbone"], "empty backbone state_dict"
pcfg = chk["config"]["projector"]
w = chk["projector"]["net.3.weight"]
assert tuple(w.shape) == (pcfg["out_dim"], pcfg["hidden_dim"]), (
    f"projector out layer {tuple(w.shape)} != (out_dim, hidden_dim) "
    f"({pcfg['out_dim']}, {pcfg['hidden_dim']}) — check PROJECTOR"
)
print("OK — backbone.pt is well-formed and loadable by extract.py / quality_gate.py")
print("  projector out layer:", tuple(w.shape), "(out_dim, hidden_dim)")

In [ ]:
# Colab only: download the result. Elsewhere, grab backbone.pt from the file browser.
try:
    from google.colab import files  # type: ignore
    files.download(OUT_PATH)
except ImportError:
    import os
    print("saved at:", os.path.abspath(OUT_PATH))